# 03 Model Preparation

## Objective

Prepare the dataset for the upcoming machine learning stage based on the findings from EDA and feature review.

This notebook will:
- Load the dataset
- Review the target variable
- Identify usable predictor features
- Check data quality before modeling
- Examine class distribution
- Prepare a modeling dataset without making final preprocessing decisions



In [25]:
import pandas as pd

DATA_PATH = "../data/raw/diabetic_data.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Number of columns:", len(df.columns))


Dataset shape: (101766, 50)
Number of columns: 50


In [26]:
# Create binary target for 30-day readmission
df["readmitted_30"] = (df["readmitted"] == "<30").astype(int)

print("Target distribution:")
print(df["readmitted_30"].value_counts())

print("\nTarget proportions:")
print(df["readmitted_30"].value_counts(normalize=True))

Target distribution:
readmitted_30
0    90409
1    11357
Name: count, dtype: int64

Target proportions:
readmitted_30
0    0.888401
1    0.111599
Name: proportion, dtype: float64


In [27]:
# Separate target from predictor features

TARGET = "readmitted_30"

X = df.drop(columns=[TARGET])
y = df[TARGET]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nTarget column:", TARGET)

X shape: (101766, 50)
y shape: (101766,)

Target column: readmitted_30


In [28]:
print("Predictor columns:")
print(X.columns.tolist())

print("\nNumber of predictor columns:", len(X.columns))

Predictor columns:
['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']

Number of predictor columns: 50


## Preliminary Feature Review

Based on the EDA and feature review:

- `encounter_id` and `patient_nbr` are identifiers and should not be used directly as model predictors.
- `readmitted` is the original target column and should not be used as a predictor because `readmitted_30` was derived from it.
- `discharge_disposition_id` requires special consideration because it contains discharge-related information.
- High-missingness and categorical features will be handled according to the team's agreed preprocessing plan.
- Diagnosis and medication features require appropriate categorical representation.

In [29]:
# Identify columns requiring special handling

id_columns = ["encounter_id", "patient_nbr"]
original_target = ["readmitted"]
timing_sensitive = ["discharge_disposition_id"]

print("ID columns:", id_columns)
print("Original target:", original_target)
print("Timing-sensitive feature:", timing_sensitive)

ID columns: ['encounter_id', 'patient_nbr']
Original target: ['readmitted']
Timing-sensitive feature: ['discharge_disposition_id']


In [30]:
# Check repeated patient encounters

unique_patients = df["patient_nbr"].nunique()
total_encounters = len(df)

print("Total encounters:", total_encounters)
print("Unique patients:", unique_patients)
print("Repeated patient encounters:", total_encounters - unique_patients)

Total encounters: 101766
Unique patients: 71518
Repeated patient encounters: 30248


In [31]:
# Distribution of encounters per patient

encounter_counts = df["patient_nbr"].value_counts()

print("Patients with more than one encounter:",
      (encounter_counts > 1).sum())

print("\nMaximum encounters for one patient:",
      encounter_counts.max())

Patients with more than one encounter: 16773

Maximum encounters for one patient: 40


In [32]:
# Inspect the distribution of encounters per patient

print(encounter_counts.describe())

count    71518.000000
mean         1.422942
std          1.090740
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         40.000000
Name: count, dtype: float64


In [33]:
# Show the most frequent encounter counts

print("\nEncounter count distribution:")
print(encounter_counts.value_counts().sort_index().head(15))


Encounter count distribution:
count
1     54745
2     10434
3      3328
4      1421
5       717
6       346
7       207
8       111
9        70
10       42
11       20
12       19
13       14
14        5
15        9
Name: count, dtype: int64


In [34]:
# Check target outcomes among patients with multiple encounters

patient_target_counts = df.groupby("patient_nbr")["readmitted_30"].nunique()

multi_encounter_patients = patient_target_counts[patient_target_counts > 1]

print("Patients with multiple encounters:", (encounter_counts > 1).sum())
print("Patients with both target classes:", len(multi_encounter_patients))

Patients with multiple encounters: 16773
Patients with both target classes: 6481


In [35]:
# Show a few patients with different target outcomes

example_patients = multi_encounter_patients.head(10).index

print(
    df[df["patient_nbr"].isin(example_patients)]
    [["patient_nbr", "encounter_id", "readmitted_30"]]
    .sort_values("patient_nbr")
)

       patient_nbr  encounter_id  readmitted_30
4267           135      24437208              1
4780           135      26264286              0
15848         1314      60254142              0
19765         1314      70190028              1
19914         1314      70601076              0
13674        11394      54021606              1
18390        11394      66624354              0
7665         11394      35935938              0
7923         11511      36676056              0
5302         11511      27873216              1
5529         11511      28600458              0
5123         11511      27315318              1
21385        13041      74513244              0
19259        13041      68799558              0
8291         13041      37761462              0
3058         13041      19423404              0
2763         13041      17719134              1
2529         13041      16429764              1
5196         13041      27586146              0
630          13041       4971204        

In [36]:
# Count patients with multiple encounters and different target outcomes

multi_encounter_mask = encounter_counts > 1

patients_with_multiple_encounters = encounter_counts[multi_encounter_mask].index

multi_patient_target_counts = (
    df[df["patient_nbr"].isin(patients_with_multiple_encounters)]
    .groupby("patient_nbr")["readmitted_30"]
    .nunique()
)

patients_with_both_outcomes = (
    multi_patient_target_counts == 2
).sum()

print("Patients with multiple encounters:",
      len(patients_with_multiple_encounters))

print("Patients with both readmission outcomes:",
      patients_with_both_outcomes)

print("Percentage of multi-encounter patients with both outcomes:",
      round(
          patients_with_both_outcomes /
          len(patients_with_multiple_encounters) * 100,
          2
      ), "%")

Patients with multiple encounters: 16773
Patients with both readmission outcomes: 6481
Percentage of multi-encounter patients with both outcomes: 38.64 %


## Patient-Level Split Consideration

The dataset contains repeated encounters for the same patient.

- 101,766 total encounters
- 71,518 unique patients
- 16,773 patients have multiple encounters
- 6,481 multi-encounter patients have both 30-day readmission outcomes
- 38.64% of multi-encounter patients have both target classes

Because repeated encounters from the same patient can have different outcomes, a simple row-level random train/test split could place encounters from the same patient in both training and test sets.

Therefore, patient-level separation should be considered for the final train/test split to reduce patient-level information leakage.

In [37]:
# Target distribution at the patient level

patient_target_summary = (
    df.groupby("patient_nbr")["readmitted_30"]
    .agg(["count", "sum", "mean"])
)

print("Number of unique patients:", len(patient_target_summary))

print("\nPatient-level target rate:")
print(patient_target_summary["mean"].describe())

Number of unique patients: 71518

Patient-level target rate:
count    71518.000000
mean         0.071123
std          0.213452
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
Name: mean, dtype: float64


## Patient-Level Target Distribution

The patient-level analysis shows:

- 71,518 unique patients
- Mean observed readmission rate per patient: 7.11%
- Median patient-level readmission rate: 0%
- 75% of patients have an observed readmission rate of 0%
- Some patients have an observed readmission rate of 100%

This indicates substantial class imbalance at the patient level. Therefore, the final train/test splitting strategy should preserve class representation while keeping encounters from the same patient separated between datasets.

## Proposed Train/Test Split Strategy

Because multiple encounters can belong to the same patient, the final dataset split should be performed at the patient level rather than by randomly splitting individual encounters.

The proposed approach is:

1. Keep `patient_nbr` available while creating the split.
2. Assign each patient entirely to either the training or test set.
3. Ensure the target distribution is reasonably represented across the two sets.
4. Remove `patient_nbr` from the model features after the patient-level split is established.
5. Perform all preprocessing transformations using the training data only, where applicable.

The exact split implementation will be finalized with the team before preprocessing and modeling.

In [38]:
# Check categorical feature cardinality

categorical_columns = df.select_dtypes(include=["object"]).columns

cardinality = (
    df[categorical_columns]
    .nunique(dropna=False)
    .sort_values(ascending=False)
)

print("Categorical feature cardinality:")
print(cardinality)

Categorical feature cardinality:
diag_3                      790
diag_2                      749
diag_1                      717
medical_specialty            73
payer_code                   18
age                          10
weight                       10
race                          6
glipizide                     4
glyburide-metformin           4
insulin                       4
miglitol                      4
acarbose                      4
rosiglitazone                 4
pioglitazone                  4
glyburide                     4
chlorpropamide                4
nateglinide                   4
repaglinide                   4
metformin                     4
glimepiride                   4
A1Cresult                     4
max_glu_serum                 4
readmitted                    3
gender                        3
tolazamide                    3
glimepiride-pioglitazone      2
diabetesMed                   2
change                        2
metformin-pioglitazone        2
metform

/var/folders/67/m53prws178q2hzd87j36jqgh0000gn/T/ipykernel_34017/3151961320.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(include=["object"]).columns


In [39]:
# Identify categorical columns with very low variation

low_cardinality = cardinality[cardinality <= 2]

print("Categorical columns with 2 or fewer unique values:")
print(low_cardinality)

Categorical columns with 2 or fewer unique values:
glimepiride-pioglitazone    2
diabetesMed                 2
change                      2
metformin-pioglitazone      2
metformin-rosiglitazone     2
acetohexamide               2
glipizide-metformin         2
troglitazone                2
tolbutamide                 2
citoglipton                 1
examide                     1
dtype: int64


In [40]:
# Check the actual values of the constant columns

constant_columns = cardinality[cardinality == 1].index.tolist()

print("\nConstant categorical columns:")
for col in constant_columns:
    print(f"{col}: {df[col].unique()}")


Constant categorical columns:
citoglipton: <StringArray>
['No']
Length: 1, dtype: str
examide: <StringArray>
['No']
Length: 1, dtype: str


In [41]:
# Missing-value review for categorical features

missing_categorical = (
    df[categorical_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("Missing values in categorical features:")
print(missing_categorical)

Missing values in categorical features:
max_glu_serum               96420
A1Cresult                   84748
acarbose                        0
miglitol                        0
troglitazone                    0
tolazamide                      0
examide                         0
citoglipton                     0
insulin                         0
race                            0
pioglitazone                    0
glyburide-metformin             0
glipizide-metformin             0
glimepiride-pioglitazone        0
metformin-rosiglitazone         0
metformin-pioglitazone          0
change                          0
diabetesMed                     0
rosiglitazone                   0
glyburide                       0
tolbutamide                     0
diag_3                          0
age                             0
weight                          0
payer_code                      0
medical_specialty               0
diag_1                          0
diag_2                          0
metformi

In [42]:
# Missing-value percentage

missing_categorical_pct = (
    df[categorical_columns]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

print("\nMissing-value percentage:")
print(missing_categorical_pct)


Missing-value percentage:
max_glu_serum               94.746772
A1Cresult                   83.277322
acarbose                     0.000000
miglitol                     0.000000
troglitazone                 0.000000
tolazamide                   0.000000
examide                      0.000000
citoglipton                  0.000000
insulin                      0.000000
race                         0.000000
pioglitazone                 0.000000
glyburide-metformin          0.000000
glipizide-metformin          0.000000
glimepiride-pioglitazone     0.000000
metformin-rosiglitazone      0.000000
metformin-pioglitazone       0.000000
change                       0.000000
diabetesMed                  0.000000
rosiglitazone                0.000000
glyburide                    0.000000
tolbutamide                  0.000000
diag_3                       0.000000
age                          0.000000
weight                       0.000000
payer_code                   0.000000
medical_specialty      

In [43]:
# Check '?' as a missing-value marker

question_mark_counts = (
    df[categorical_columns]
    .apply(lambda col: (col == "?").sum())
    .sort_values(ascending=False)
)

question_mark_pct = (
    question_mark_counts
    .div(len(df))
    .mul(100)
)

question_mark_summary = pd.DataFrame({
    "count": question_mark_counts,
    "percentage": question_mark_pct
})

print("Columns containing '?' values:")
print(question_mark_summary[question_mark_summary["count"] > 0])

Columns containing '?' values:
                   count  percentage
weight             98569   96.858479
medical_specialty  49949   49.082208
payer_code         40256   39.557416
race                2273    2.233555
diag_3              1423    1.398306
diag_2               358    0.351787
diag_1                21    0.020636


## Missing-Value Findings

The dataset uses both actual missing values (`NaN`) and the string `?` as missing/unknown markers.

### Actual NaN values

- `max_glu_serum`: 96,420 values (94.75%)
- `A1Cresult`: 84,748 values (83.28%)

### '?' values

- `weight`: 98,569 values (96.86%)
- `medical_specialty`: 49,949 values (49.08%)
- `payer_code`: 40,256 values (39.56%)
- `race`: 2,273 values (2.23%)
- `diag_3`: 1,423 values (1.40%)
- `diag_2`: 358 values (0.35%)
- `diag_1`: 21 values (0.02%)

These findings indicate that missing-value handling will be an important part of preprocessing. The treatment of each feature will be decided jointly with the team rather than independently in this notebook.

In [44]:
# Numerical feature summary

numerical_columns = df.select_dtypes(include=["number"]).columns.tolist()

# Remove the target from the predictor list for this analysis
numerical_predictors = [
    col for col in numerical_columns
    if col != "readmitted_30"
]

print("Numerical predictor columns:")
print(numerical_predictors)

print("\nNumber of numerical predictors:", len(numerical_predictors))

Numerical predictor columns:
['encounter_id', 'patient_nbr', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']

Number of numerical predictors: 13


In [45]:
# Summary statistics for numerical predictors

numerical_summary = df[numerical_predictors].describe().T

print(numerical_summary[
    ["min", "25%", "50%", "75%", "max", "mean", "std"]
])

                              min         25%          50%           75%  \
encounter_id              12522.0  84961194.0  152388987.0  2.302709e+08   
patient_nbr                 135.0  23413221.0   45505143.0  8.754595e+07   
admission_type_id             1.0         1.0          1.0  3.000000e+00   
discharge_disposition_id      1.0         1.0          1.0  4.000000e+00   
admission_source_id           1.0         1.0          7.0  7.000000e+00   
time_in_hospital              1.0         2.0          4.0  6.000000e+00   
num_lab_procedures            1.0        31.0         44.0  5.700000e+01   
num_procedures                0.0         0.0          1.0  2.000000e+00   
num_medications               1.0        10.0         15.0  2.000000e+01   
number_outpatient             0.0         0.0          0.0  0.000000e+00   
number_emergency              0.0         0.0          0.0  0.000000e+00   
number_inpatient              0.0         0.0          0.0  1.000000e+00   
number_diagn

In [46]:
# Check skewness of numerical predictors

skewness = (
    df[numerical_predictors]
    .skew()
    .sort_values(ascending=False)
)

print("Numerical feature skewness:")
print(skewness)

Numerical feature skewness:
number_emergency            22.855582
number_outpatient            8.832959
number_inpatient             3.614139
discharge_disposition_id     2.563067
admission_type_id            1.591984
num_medications              1.326672
num_procedures               1.316415
time_in_hospital             1.133999
admission_source_id          1.029935
encounter_id                 0.699142
patient_nbr                  0.471281
num_lab_procedures          -0.236544
number_diagnoses            -0.876746
dtype: float64


In [47]:
# Identify highly skewed numerical features

highly_skewed = skewness[skewness.abs() > 1]

print("\nFeatures with absolute skewness > 1:")
print(highly_skewed)



Features with absolute skewness > 1:
number_emergency            22.855582
number_outpatient            8.832959
number_inpatient             3.614139
discharge_disposition_id     2.563067
admission_type_id            1.591984
num_medications              1.326672
num_procedures               1.316415
time_in_hospital             1.133999
admission_source_id          1.029935
dtype: float64


## Numerical Feature Findings

The numerical feature analysis shows substantial skewness in several variables.

Highly skewed features include:

- `number_emergency`
- `number_outpatient`
- `number_inpatient`
- `discharge_disposition_id`
- `admission_type_id`
- `num_medications`
- `num_procedures`
- `time_in_hospital`
- `admission_source_id`

The utilization features, especially emergency, outpatient, and inpatient visit counts, are strongly right-skewed.

These findings should be considered during preprocessing and modeling. No transformations are applied in this notebook because final preprocessing decisions will be coordinated with the team.

In [48]:
# Review admission and discharge related features

timing_features = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id"
]

for col in timing_features:
    print(f"\n===== {col} =====")
    print(df[col].value_counts(dropna=False).head(15))


===== admission_type_id =====
admission_type_id
1    53990
3    18869
2    18480
6     5291
5     4785
8      320
7       21
4       10
Name: count, dtype: int64

===== discharge_disposition_id =====
discharge_disposition_id
1     60234
3     13954
6     12902
18     3691
2      2128
22     1993
11     1642
5      1184
25      989
4       815
7       623
23      412
13      399
14      372
28      139
Name: count, dtype: int64

===== admission_source_id =====
admission_source_id
7     57494
1     29565
17     6781
4      3187
6      2264
2      1104
5       855
3       187
20      161
9       125
8        16
22       12
10        8
14        2
11        2
Name: count, dtype: int64


## Admission and Discharge Feature Review

The admission-related features include:

- `admission_type_id`
- `admission_source_id`
- `discharge_disposition_id`

These features describe the circumstances surrounding the hospital encounter and should be treated carefully during preprocessing.

`discharge_disposition_id` requires particular attention because it describes the patient's disposition at discharge. Its availability and timing relative to the prediction point should be confirmed before including it in the final model.

No columns are removed at this stage. The final feature-availability decision will be made jointly by the team.

# Model Preparation Summary

## Key Findings

### Target
- 11.16% of encounters resulted in 30-day readmission.
- The target is imbalanced.

### Patient Structure
- 101,766 encounters
- 71,518 unique patients
- 16,773 patients have multiple encounters
- 6,481 multi-encounter patients have both target outcomes
- 38.64% of multi-encounter patients have both outcomes

### Missing Values
- Actual `NaN` values are especially high in `max_glu_serum` and `A1Cresult`.
- `?` is used as a missing/unknown marker in several categorical features.
- `weight`, `medical_specialty`, and `payer_code` have particularly high rates of `?`.

### Categorical Features
- Diagnosis features have high cardinality.
- `medical_specialty` also has relatively high cardinality.
- `citoglipton` and `examide` are constant columns.

### Numerical Features
- Several utilization features are strongly right-skewed.
- Numerical features have substantially different ranges.

### Modeling Considerations
- Patient-level separation should be considered for train/test splitting.
- Identifier columns should not be used directly as model predictors.
- The original `readmitted` column should not be used as a predictor because the modeling target is `readmitted_30`.
- Missing-value handling, encoding, scaling, feature selection, and final feature availability will be decided jointly with the team.

## Conclusion

This notebook documents the findings needed to guide the preprocessing and modeling stages. No final preprocessing or feature-selection decisions are made here.